# Customer Churn Prediction

**Goal:** Build and compare machine learning models to predict which telecom customers are likely to churn, enabling proactive retention campaigns.

**Dataset:** Synthetic Telco-style dataset (7,043 customers, 20 features)  
**Target:** `Churn` — binary (Yes / No)

---
### Notebook Structure
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training & Comparison
5. Best Model Evaluation
6. Feature Importance & SHAP
7. Business Insights & Recommendations

## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')  # so src/ is importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score

from src.data_generator import generate_churn_data
from src.preprocessing import build_preprocessor, split_data
from src.models import build_pipeline, get_base_models
from src.evaluation import (
    compute_metrics, metrics_table,
    plot_churn_distribution, plot_confusion_matrices,
    plot_roc_curves, plot_metrics_comparison,
    plot_feature_importance, plot_shap_summary,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

In [ ]:
df = generate_churn_data(n_samples=7043, random_state=42)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. Exploratory Data Analysis

### 2.1 Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
print(churn_counts)
print(f"\nChurn rate: {churn_counts['Yes'] / len(df):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=['#4c72b0', '#dd8452'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#4c72b0', '#dd8452'], startangle=90)
axes[1].set_title('Churn Rate')
plt.suptitle('Target Variable Distribution', fontsize=13)
plt.tight_layout()
plt.show()

> **Observation:** The dataset is imbalanced (~26% churn). We'll use class-weighted models and evaluate with AUC-ROC rather than accuracy alone.

### 2.2 Numerical Features vs Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    for label, color in zip(['No', 'Yes'], ['#4c72b0', '#dd8452']):
        subset = df[df['Churn'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={label}')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend()
plt.suptitle('Numerical Features by Churn', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    df.boxplot(column=col, by='Churn', ax=ax, 
               boxprops=dict(color='#4c72b0'),
               medianprops=dict(color='#dd8452', linewidth=2))
    ax.set_title(col)
    ax.set_xlabel('Churn')
plt.suptitle('')
plt.tight_layout()
plt.show()

### 2.3 Categorical Features vs Churn

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender',
            'SeniorCitizen', 'Partner', 'Dependents', 'PaperlessBilling']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452'], legend=False)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Churn %')
    ax.tick_params(axis='x', rotation=30)
    ax.axhline(y=df['Churn'].value_counts(normalize=True)['Yes']*100, 
               color='gray', linestyle='--', linewidth=0.8, label='Avg')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles[:2], ['No Churn', 'Churn'], loc='upper right')
plt.suptitle('Churn Rate by Categorical Features', fontsize=13)
plt.tight_layout()
plt.show()

### 2.4 Correlation Heatmap (Numerical)

In [ ]:
df_num = df[num_cols + ['SeniorCitizen']].copy()
df_num['Churn'] = (df['Churn'] == 'Yes').astype(int)

corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

### 2.5 Churn Rate by Tenure Group

In [ ]:
df['tenure_group'] = pd.cut(
    df['tenure'], bins=[0, 12, 24, 36, 48, 72],
    labels=['0-12 mo', '13-24 mo', '25-36 mo', '37-48 mo', '49-72 mo']
)
churn_by_tenure = df.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

fig, ax = plt.subplots(figsize=(8, 4))
churn_by_tenure.plot(kind='bar', ax=ax, color='#dd8452', edgecolor='white')
ax.axhline(y=churn_by_tenure.mean(), color='#4c72b0', linestyle='--', label='Overall avg')
ax.set_title('Churn Rate by Tenure Group')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Tenure Group')
ax.tick_params(axis='x', rotation=0)
ax.legend()
for i, v in enumerate(churn_by_tenure):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df.drop(columns=['tenure_group']))

print(f'Train : {X_train.shape[0]:,} rows')
print(f'Val   : {X_val.shape[0]:,} rows')
print(f'Test  : {X_test.shape[0]:,} rows')
print(f'\nChurn rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}')

In [ ]:
preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
print(f'After preprocessing: {X_train_t.shape[1]} features')
try:
    print('Feature names:', list(preprocessor.get_feature_names_out())[:10], '...')
except:
    pass

## 4. Model Training & Comparison

In [ ]:
base_models = get_base_models()
print('Models to train:', list(base_models.keys()))

In [ ]:
pipelines = {}
results = {}

for name, estimator in base_models.items():
    print(f'Training {name} ...', end=' ', flush=True)
    pipe = build_pipeline(build_preprocessor(), estimator)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    results[name] = compute_metrics(y_test, y_pred, y_prob)
    pipelines[name] = pipe
    print(f"AUC={results[name]['ROC-AUC']:.4f}")

In [ ]:
df_results = metrics_table(results)
df_results.style.background_gradient(cmap='YlOrRd', axis=0)

In [ ]:
plot_metrics_comparison(results, save=False)

## 5. Best Model Evaluation

In [ ]:
best_name = df_results.index[0]
best_pipe = pipelines[best_name]
print(f'Best model: {best_name}')
print(f"AUC-ROC: {df_results.loc[best_name, 'ROC-AUC']:.4f}")

In [ ]:
y_pred_best = best_pipe.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn']))

In [ ]:
plot_roc_curves(pipelines, X_test, y_test, save=False)

In [ ]:
plot_confusion_matrices(pipelines, X_test, y_test, save=False)

### 5.1 Cross-Validation (5-fold)

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'{best_name} — 5-Fold CV AUC-ROC')
print(f'  Mean: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')
print(f'  Scores: {[round(s, 4) for s in cv_scores]}')

## 6. Feature Importance & SHAP

In [ ]:
plot_feature_importance(best_pipe, list(X_train.columns), best_name, top_n=20, save=False)

In [ ]:
X_shap = X_test.sample(500, random_state=42)
plot_shap_summary(best_pipe, X_shap, best_name, save=False)

## 7. Business Insights & Recommendations

### Key Churn Drivers

| Driver | Impact | Action |
|---|---|---|
| **Month-to-month contract** | High risk | Offer discounts to switch to annual plans |
| **Short tenure (< 12 months)** | Very high risk | Strengthen onboarding; early check-in calls |
| **Fiber optic internet** | Elevated risk | Investigate service quality/price perception |
| **Electronic check payment** | Moderate risk | Encourage auto-pay with incentive |
| **No online security / tech support** | Elevated risk | Bundle value-added services |
| **Senior citizen** | Moderate risk | Dedicated senior support line |

### Recommended Actions

1. **Proactive outreach** — Score all active customers monthly; flag top 20% as high-risk.
2. **Early-tenure programme** — Assign a customer success rep to customers in month 1–12.
3. **Contract upgrade campaign** — Offer 1-month free on 12-month plan to month-to-month customers.
4. **Bundle add-ons** — Auto-include 3-month free Online Security for new Fiber customers.
5. **Payment incentive** — $5/month discount for auto-pay enrollment.

### Expected Business Impact

Assuming an average customer lifetime value of \$1,200 and 1,000 high-risk customers identified:
- Retaining 20% of flagged customers → **\$240,000 saved per campaign cycle**
- Model precision ensures retention spend targets genuinely at-risk customers

In [ ]:
# --- Demonstration: score new customers ---
import joblib, os

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(best_pipe, '../outputs/models/best_model.pkl')

new_customers = df.drop(columns=['Churn', 'tenure_group'], errors='ignore').head(5)
from src.preprocessing import load_and_clean
new_clean = load_and_clean(new_customers)

probs = best_pipe.predict_proba(new_clean)[:, 1]
preds = best_pipe.predict(new_clean)

result_df = new_customers[['customerID']].copy()
result_df['churn_probability'] = probs.round(3)
result_df['churn_prediction'] = np.where(preds == 1, 'At Risk', 'Retained')
result_df['risk_tier'] = pd.cut(
    probs, bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High']
)
print(result_df.to_string(index=False))

---
**Notebook complete.** Model saved to `outputs/models/best_model.pkl`.